v61  
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan


v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


v57, v58  
added marco subplotsThe macro regime framework is now fully documented with:
- Trend (SMA200 deviation) → Where we are in the cycle  
- Trend Velocity (Z) → How fast we're moving relative to normal
- VIX-Z → Market fear/complacency levels  

v56  

- De-coupled features_df and macro_df
- generate_features and audit_feature_engineering_integrity use GLOBAL_SETTINGS

v55  
Added
- audit_feature_engineering_integrity (check calculation in features_df)  

These are the metrics in plot  
- --- 1. LEGACY / SANITY CHECKS ---
- "Price Gain": lambda obs: QuantUtils.calculate_gain(obs["lookback_close"]),
- "Sharpe": lambda obs: QuantUtils.calculate_sharpe(obs["lookback_returns"]),
- "Sharpe (ATRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["atrp"]
    ),
- "Sharpe (TRP)": lambda obs: QuantUtils.calculate_sharpe_vol(
        obs["lookback_returns"], obs["trp"]
    ),
- --- 2. NEW QUANT METRICS ---
- "Momentum (21d)": lambda obs: obs["mom_21"],
- "Information Ratio (IR)": lambda obs: obs["ir_63"],  # Kept this one
- "Consistency (WinRate)": lambda obs: obs["consistency"],
- "Oversold (RSI)": lambda obs: -obs["rsi"],
- "Dip Buyer (Drawdown)": lambda obs: -obs["dd_21"],
- "Low Volatility": lambda obs: -obs["atrp"],







v54
-  **Replaced plot_walk_forward_analyzer with create_walk_forward_analyzer**


v53  
Looking at this registry with a quant lens, the list is **comprehensive but bloated**—we have **momentum measured five times under different names** (roc₁, roc₃, roc₅, roc₁₀, roc₂₁ and their negative twins “Pullback”).  
That’s **10 slots** telling us almost the same story at slightly different lags; in a rank-based engine they will **crowd the signal space** and inflate turnover without adding IC.

Duplicate / redundant cluster  
- Momentum 1 D ↔ Pullback 1 D (perfect mirror)  
- Same for 3 D, 5 D, 10 D, 21 D.  
**Keep one side only**—momentum is enough; the portfolio constructor can always **reverse the rank** if it wants “oversold”.

Close cousins that can be merged  
- “Sharpe” vs “Sharpe (ATRP)” – both are return / vol; keep **ATRP version** because it is regime-aware and smoother.  
- “RVol” vs “Vol_Regime” – both capture vol expansion; keep the **longer-memory one** (Vol_Regime) and drop the intraday snapshot.

Gaps that matter to a quant  
1. **Consistency sensor**: nowhere do we ask “how often did the ticker close higher than it opened?” – add **5-day win-rate** or **up-day hit-ratio**.  
2. **Risk-adjusted intraday strength**: no **Sharpe(on-balance volume)** or **volume-momentum efficiency**; OBV_Score is raw, not risk-scaled.  
3. **Benchmark-relative consistency**: “Alpha (RelStrength)” is cumulative; add **rolling information ratio vs SPY** to catch *sustained* alpha, not one gap.  
4. **Tail flag**: no **skew** or **max-drawdown** metric; a single 20 % gap stock can poison the book.  
5. **Macro regime overlay**: no **beta-to-SPY** or **correlation-break** sensor; mid-2022 macro swings showed that low-beta names behaved like a different asset class.

Recommended minimal clean set (≤ 12 metrics)

1. Sharpe(ATRP) – strategic anchor  
2. Momentum 21 D – slow trend  
3. Momentum 5 D – fast trend  
4. 5-day win-rate – consistency  
5. RSI(Trend) – strength confirmation  
6. OBV_Score – volume conviction  
7. Vol_Regime – vol expansion filter  
8. Alpha(RelStrength) 63-day IR – benchmark consistency  
9. Max 21-day drawdown – tail guard  
10. Beta-to-SPY – macro regime tag  

Drop everything else; the freed-up slots reduce collinearity, cut turnover, and leave head-room for **interaction terms** (e.g. momentum × consistency) that actually add orthogonal signal.



Below is a single, fully-vectorised block that adds the **five gap metrics** to your existing MultiIndex OHLCV frame.  
It never loops over tickers; everything is done with `groupby(level=0).rolling(...)` so it runs in C-speed and keeps the same index.

```python
import pandas as pd
import numpy as np

# ----------  CONFIG  -------------------------------------------------
LKB_RET   = 21          # look-back for return-based metrics
LKB_CONS  = 5           # consistency window (days)
LKB_IR    = 63          # IR window
LKB_BETA  = 63          # beta window
LKB_TAIL  = 21          # max-drawdown window
BENCH     = 'SPY'       # ticker that exists in your universe
# ---------------------------------------------------------------------

# 1.  DAILY RETURNS ----------------------------------------------------
df['ret'] = df.groupby(level=0)['Adj Close'].pct_change()

# 2.  CONSISTENCY SENSOR  (5-day win-rate) -----------------------------
df['up']  = df['ret'].gt(0).astype(int)
df['consistency_5d'] = (df.groupby(level=0)['up']
                          .rolling(LKB_CONS).mean()
                          .reset_index(level=0, drop=True))

# 3.  BENCHMARK-RELATIVE CONSISTENCY  (63-day IR vs SPY) ---------------
# need benchmark return
bench_ret = df.xs(BENCH, level=0)['ret'].rename('bench_ret')
df = df.join(bench_ret, how='left')          # broadcast to all tickers

df['active'] = df['ret'] - df['bench_ret']
g = df.groupby(level=0)
active_mean  = g['active'].rolling(LKB_IR).mean()
active_std   = g['active'].rolling(LKB_IR).std()
df['IR_63d'] = active_mean / active_std      # Information Ratio

# 4.  TAIL FLAG  (21-day max drawdown) ---------------------------------
roll_max = g['Adj Close'].rolling(LKB_TAIL).max()
dd = (df['Adj Close'] - roll_max) / roll_max
df['max_dd_21d'] = dd.groupby(level=0).rolling(LKB_TAIL).min()

# 5.  MACRO REGIME OVERLAY  (beta to SPY) ------------------------------
cov  = g['ret'].rolling(LKB_BETA).cov(df['bench_ret'])
var  = df['bench_ret'].groupby(level=0).rolling(LKB_BETA).var()
df['beta_SPY'] = cov / var

# 6.  RISK-ADJUSTED INTRADAY STRENGTH  (OBV Sharpe) --------------------
# OBV
df['close_chg'] = df.groupby(level=0)['Adj Close'].diff()
df['vol_dir']   = np.where(df['close_chg'] > 0,  df['Volume'],
                   np.where(df['close_chg'] < 0, -df['Volume'], 0))
df['obv'] = df.groupby(level=0)['vol_dir'].cumsum()

# OBV return & vol
df['obv_ret'] = df.groupby(level=0)['obv'].pct_change()
obv_mean = g['obv_ret'].rolling(LKB_RET).mean()
obv_std  = g['obv_ret'].rolling(LKB_RET).std()
df['OBV_Sharpe_21d'] = obv_mean / obv_std

# drop helper columns --------------------------------------------------
df.drop(columns=['up','bench_ret','active','close_chg','vol_dir'], inplace=True)
```

After the block you have five new columns:

- `consistency_5d`      – 5-day win-rate (0-1)  
- `IR_63d`              – 63-day Information Ratio vs SPY  
- `max_dd_21d`          – 21-day maximum drawdown (≤ 0)  
- `beta_SPY`            – rolling beta to SPY  
- `OBV_Sharpe_21d`      – OBV risk-adjusted momentum  

All are aligned to the original MultiIndex and ready to be ranked or z-scored inside your Alpha Engine.

v52  
- **Cascase Filter results `AGREED` with bot_v54i.ipynb**
- **Cascade Filter works with df_ohlcv_subset**
- **verify_engine_results_short_form**
- **verify_engine_results_long_form**
-  **The Temporal Alignment Fix:** We synchronized the "Reward" (Returns) and "Risk" (Volatility) by implementing the $N-1$ denominator logic. This ensures that Day 1's volatility no longer dilutes your Sharpe scores.
-  **The Event-Driven Re-normalization:** We verified that the Engine correctly resets capital and weights at the start of the Holding period, giving you an accurate "Fresh Start" performance metric.
-  **The Double-Blind Verification:** We proved that the Engine's True Range (TRP) math is flawless by recreating it from raw High/Low/Close data and achieving an 8-decimal match.
-  **Mathematical Fortification:** We centralized all logic into a polymorphic `QuantUtils` kernel that handles both single-portfolio reports and whole-universe rankings with built-in numerical safety.
-  **Volatility Evolution:** We successfully added `TRP` (True Range Percent) and the `Sharpe (TRP)` metric, giving you a raw, high-frequency alternative to the smoothed ATR.
-  **Data Integrity:** We implemented the "Momentum Collapse" tripwire (`verify_ranking_integrity`) to ensure that your risk-adjusted rankings never accidentally devolve into simple price momentum.
-  **The "Audit Pack" Architecture:** We collapsed fragmented results into a single, atomic container, ensuring that your inputs, results, and debug data are always perfectly synchronized.
-  **Total Transparency:** We replaced scattered CSV files with a unified **Excel Audit Report**, allowing for 1-to-1 manual verification of every calculation in the system.



v51

UNDO v50, Calculate Sharpe(ATR) using mean over lookback period.  

Comment out ``# --- PINPOINT START: ATRP SWITCH ---`` in function ``_select_tickers`` can switch between ``Averaged ATRP over lookback period`` and ``Current ATRP``  
    # --- PINPOINT START: ATRP SWITCH ---  
    # To switch between Old (Averaged ATRP) and New (Current ATRP):  
    # 1. Comment out the logic you DON'T want.  
    # 2. Uncomment the logic you DO want.  


v50

Ticker selection based on atrp_value_for_obs based on decision day, was based on average over lookback period. 

v48  
### Summary of what you just accomplished:
1.  **Strict Math:** `QuantUtils` now contains an `assert` that prevents any dev (or AI) from filling the first day with 0.0.
2.  **Semantic Protection:** Variables are now named `returns_WITH_BOUNDARY_NAN`, signaling to the AI that the Null value is part of its identity.
3.  **Complete SOLID Separation:** The Engine CONDUCTS the simulation, while `QuantUtils` CALCULATES the results. They no longer share logic.

**1. Data Flow of `plot_walk_forward_analyzer`**
The function acts as a **UI wrapper** around the `AlphaEngine` class. The flow is:
1.  **Input:** User selects parameters (Dates, Lookback, Strategy).
2.  **State Construction:** `AlphaEngine` slices the historical data (`df_ohlcv`, `df_atrp`) up to the `decision_date`.
3.  **Policy Execution (Hardcoded):** The engine applies the logic (e.g., `METRIC_REGISTRY['Sharpe']`) to rank stocks based *only* on the Lookback window.
4.  **Environment Step:** It simulates a "Buy" at `decision_date + 1` and calculates the returns over the `holding_period`.
5.  **Reward Generation:** It outputs performance metrics (`holding_p_gain`, `holding_p_sharpe`).

In [32]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
import importlib

modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import os
import numpy as np

from IPython.display import display
from dataclasses import fields, asdict, is_dataclass
from typing import List, Union, Tuple 


# 4. Fresh imports (these will re-import from disk due to cache clearing above)

from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack
from core.engine import AlphaEngine
from core.features import generate_features
from core.paths import OUTPUT_DIR
# from core.performance import PerformanceCalculator
# from core.quant import QuantUtils
# from core.result import TaskResult
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
# from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# # 6. Instantiate engine (customize DataFrames as needed)
# master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output



In [2]:
data_path = r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet"

df_indices = pd.read_parquet(data_path, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close  Volume
Ticker Date                                                      
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00       0
       1992-11-23   1458.40   1458.40  1458.40    1458.40       0
       1992-11-24   1467.90   1467.90  1467.90    1467.90       0
       1992-11-25   1459.00   1459.00  1459.00    1459.00       0
       1992-11-26   1458.90   1458.90  1458.90    1458.90       0
...                     ...       ...      ...        ...     ...
^VIX3M 2026-03-03     24.43     26.34    22.78      23.54       0
       2026-03-04     23.00     23.48    21.84      22.29       0
       2026-03-05     23.01     24.94    22.43      23.86       0
       2026-03-06     26.08     27.75    24.99      27.56       0
       2026-03-09     28.04     29.09    24.86      25.34       0

[144365 rows x 5 columns]


In [ ]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

In [5]:
print(f"Takes about 1.5 minutes")

data_path = (
    r"c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet"
)

df_ohlcv = pd.read_parquet(data_path, engine="pyarrow")

Takes about 1.5 minutes


In [ ]:
print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

In [ ]:
print(f"Takes about 2.5 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 2.5 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

In [ ]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [7]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1586, Days: 16153
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [8]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [9]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 940 stocks passed filters on 2025-12-10


In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [ ]:
_tic = "^VIX"
_date = "2026-02-24"
df_indices.loc["^VIX"].tail(10)

In [34]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 10))
[  4]   📂 tickers (len=10)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]   📈 initial_weights (shape=(10,))
[ 16]   📂 perf_metrics (len=24)
[ 17]     🔢 full_p_gain (float)
[ 18]     🔢 full_p_sharpe (float)
[ 19]     🔢 full_p_sharpe_atrp (float)
[ 20]     🔢 full_p_sharpe_trp (float)
[ 21]     🔢 lookback_p_gain (float)
[ 22]     🔢 lookback_p_sharpe (float)
[ 23]     🔢 lookback_p_sharpe_atrp (float)
[ 24]     🔢 lookback_p_sharpe_trp (float)
[ 25]     🔢 holding_p_gain (float)
[ 26]     🔢 holding_p_sharpe (float)
[ 27]     🔢 holding_p_sharpe_atrp (float)
[ 28]     🔢 holding_p_sharpe_tr

In [ ]:
SU.peek(58, my_res)

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [35]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")


🕵️  NUCLEAR FEATURE AUDIT | Mode: LAST_RUN | Tickers: 10
STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | ✅ PASS
STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... DONE (0.64s)

Metric               | Max Delta    | Correlation  | Status
-------------------------------------------------------------------------------------
Ret_1d               |   0.0000e+00 |     1.000000 | ✅ PASS
ATR                  |   0.0000e+00 |     1.000000 | ✅ PASS
ATRP                 |   0.0000e+00 |     1.000000 | ✅ PASS
TRP                  |   0.0000e+00 |     1.000000 | ✅ PASS
RSI                  |   0.0000e+00 |     1.000000 | ✅ PASS
Mom_21               |   0.0000e+00 |     1.000000 | ✅ PASS
Consistency          |   0.0000e+00 |     1.000000 | ✅ PASS
DD_21                |   0.0000e+00 |     1.000000 | ✅ PASS
RollingStalePct      |   0.0000e+00 |     1.000000 | ✅ PASS
RollMedDollarVol     |   0.0000e+00 |     1.000000 | ✅ PASS
RollingSameVolCount  |   0.0000e+00 |     1.000000 | ✅ PASS


In [36]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="system")


🕵️  NUCLEAR FEATURE AUDIT | Mode: SYSTEM | Tickers: 1586
STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | ✅ PASS
STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... DONE (70.39s)

Metric               | Max Delta    | Correlation  | Status
-------------------------------------------------------------------------------------
Ret_1d               |   0.0000e+00 |     1.000000 | ✅ PASS
ATR                  |   0.0000e+00 |     1.000000 | ✅ PASS
ATRP                 |   0.0000e+00 |          nan | ✅ PASS
TRP                  |   0.0000e+00 |          nan | ✅ PASS
RSI                  |   0.0000e+00 |     1.000000 | ✅ PASS
Mom_21               |   0.0000e+00 |          nan | ✅ PASS
Consistency          |   0.0000e+00 |     1.000000 | ✅ PASS
DD_21                |   0.0000e+00 |     1.000000 | ✅ PASS
RollingStalePct      |   0.0000e+00 |     1.000000 | ✅ PASS
RollMedDollarVol     |   0.0000e+00 |     1.000000 | ✅ PASS
RollingSameVolCount  |   0.0000e+00 |     1.000000 | ✅ PASS

In [ ]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

### Export Ticker's OHLCV and its Features

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

### Audit features_df

In [ ]:
@staticmethod
def audit_feature_engineering_integrity(analyzer, df_indices=None, mode="last_run"):
    """
    # Usage to check last run, takes about 4 sec.
    audit_feature_engineering_integrity(analyzer2, mode="last_run")
    # Usage to check all df_ohlcv tickers, takes over 4 minutes (i.e. One-time "Nuclear" System Sanity Check)
    audit_feature_engineering_integrity(analyzer2, df_indices=df_indices, mode="system")
    """
    import time
    import numpy as np
    import warnings

    # 0. PULL SETTINGS FROM GLOBAL_SETTINGS (or analyzer.engine.settings if stored there)
    # This ensures the auditor uses the EXACT same rules as the engine
    atr_p = GLOBAL_SETTINGS["atr_period"]
    rsi_p = GLOBAL_SETTINGS["rsi_period"]
    win_5 = GLOBAL_SETTINGS["5d_window"]
    win_21 = GLOBAL_SETTINGS["21d_window"]
    win_63 = GLOBAL_SETTINGS["63d_window"]
    q_win = GLOBAL_SETTINGS["quality_window"]
    q_min = GLOBAL_SETTINGS["quality_min_periods"]

    start_time = time.time()
    engine = analyzer.engine
    features_df = engine.features_df
    df_ohlcv = engine.df_ohlcv_raw

    # 1. Scope Selection
    if mode == "last_run" and analyzer.last_run:
        audit_tickers = analyzer.last_run.tickers
        features_to_audit = features_df.loc[pd.IndexSlice[audit_tickers, :], :]
        ohlcv_to_audit = df_ohlcv.loc[pd.IndexSlice[audit_tickers, :], :]
    else:
        audit_tickers = features_df.index.get_level_values(0).unique()
        features_to_audit = features_df
        ohlcv_to_audit = df_ohlcv

    print(f"\n{'='*95}")
    print(
        f"🕵️  NUCLEAR FEATURE AUDIT | Mode: {mode.upper()} | Tickers: {len(audit_tickers)}"
    )
    print(f"{'='*95}")

    # STEP 1: BOUNDARY INTEGRITY
    leaks = features_to_audit.groupby(level=0).head(1)["Ret_1d"].dropna().count()
    leak_status = "✅ PASS" if leaks == 0 else f"❌ FAIL ({leaks} leaks)"
    print(f"STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | {leak_status}")

    # STEP 2: SHADOW CALCULATION
    print(
        f"STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... ", end="", flush=True
    )

    adj_close = ohlcv_to_audit["Adj Close"]
    adj_high = ohlcv_to_audit["Adj High"]
    adj_low = ohlcv_to_audit["Adj Low"]
    volume = ohlcv_to_audit["Volume"]

    shadow_data = {}

    # A. Returns & Basics
    # Explicitly turns division-by-zero results (`inf`) into `NaN`
    # Replace [np.inf, -np.inf] with np.nan
    shadow_data["shadow_Ret_1d"] = (
        adj_close.groupby(level=0).pct_change().replace([np.inf, -np.inf], np.nan)
    )

    prev_close = adj_close.groupby(level=0).shift(1)
    tr = pd.concat(
        [
            adj_high - adj_low,
            (adj_high - prev_close).abs(),
            (adj_low - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1, skipna=False)

    # B. Smoothing (ATR/RSI) - Use transform for speed and index matching
    shadow_data["shadow_ATR"] = tr.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / atr_p, adjust=False).mean()  # Replaced 14
    )

    shadow_data["shadow_ATRP"] = shadow_data["shadow_ATR"] / adj_close
    shadow_data["shadow_TRP"] = tr / adj_close

    # Auditor Step 2B - Shadow RSI with correct Inf/NaN handling
    delta = adj_close.groupby(level=0).diff()
    up, down = delta.clip(lower=0), (-delta).clip(lower=0)

    # Match Wilder's spec correctly:
    roll_up = up.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / rsi_p, adjust=False).mean()  # Replaced 14
    )
    roll_down = down.groupby(level=0).transform(
        lambda x: x.ewm(alpha=1 / rsi_p, adjust=False).mean()  # Replaced 14
    )

    # FIX: Allow division by zero (i.e. no down day) to create inf (correct RSI=100),
    # inf→100, -inf→0, NaN→50
    # then clean up remaining NaNs (initial periods/no movement)
    # - Initial periods: Before the 14-day lookback is filled, the EWM mean is undefined → NaN.
    # - Flat prices: If price doesn't move (Avg Up = 0 and Avg Down = 0), RS is 0/0 → NaN.
    # - By convention, RSI is set to 50 (neutral) when there is no directional momentum.
    rs = roll_up / roll_down  # Keep zero denominator → inf
    raw_rsi = 100 - (100 / (1 + rs))
    shadow_data["shadow_RSI"] = raw_rsi.replace({np.inf: 100, -np.inf: 0}).fillna(50)

    # C. Momentum & Consistency
    shadow_data[f"shadow_Mom_{win_21}"] = adj_close.groupby(level=0).pct_change(win_21)
    pos_ret = (shadow_data["shadow_Ret_1d"] > 0).astype(float)
    shadow_data["shadow_Consistency"] = pos_ret.groupby(level=0).transform(
        lambda x: x.rolling(win_5).mean()
    )

    # D. Risk (Beta & IR)
    if df_indices is not None:
        try:
            # USE THIS: Pull the single source of truth from the engine
            mkt_ret = engine.macro_df["Mkt_Ret"]
            # Map it to the audit tickers
            mkt_series = mkt_ret.reindex(
                ohlcv_to_audit.index.get_level_values(1)
            ).values
            mkt_series = pd.Series(mkt_series, index=ohlcv_to_audit.index)

            # Shadow Beta
            s_ret = shadow_data["shadow_Ret_1d"]
            shadow_data[f"shadow_Beta_{win_63}"] = (
                s_ret.groupby(level=0)
                .transform(
                    lambda x: x.rolling(win_63).cov(
                        mkt_ret.reindex(x.index.get_level_values(1))
                    )
                    / mkt_ret.reindex(x.index.get_level_values(1)).rolling(win_63).var()
                )
                .fillna(1.0)
            )

            # Shadow IR
            active_ret = s_ret - mkt_series
            shadow_data["shadow_IR_63"] = (
                active_ret.groupby(level=0)
                .transform(lambda x: x.rolling(win_63).mean() / x.rolling(win_63).std())
                .fillna(0.0)
            )

        except Exception as e:
            print(f" (Macro Shadow Error: {e}) ", end="")

    # E. Drawdown & Quality
    roll_max_21 = adj_close.groupby(level=0).transform(
        lambda x: x.rolling(win_21).max()
    )
    shadow_data[f"shadow_DD_{win_21}"] = (adj_close / roll_max_21 - 1).fillna(0.0)
    stale_mask = ((volume == 0) | (adj_high == adj_low)).astype(int)

    shadow_data["shadow_RollingStalePct"] = stale_mask.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).mean()
    )
    dollar_vol = adj_close * volume
    shadow_data["shadow_RollMedDollarVol"] = dollar_vol.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).median()  # Replaced 252, 126
    )

    same_vol = (volume.groupby(level=0).diff() == 0).astype(int)
    shadow_data["shadow_RollingSameVolCount"] = same_vol.groupby(level=0).transform(
        lambda x: x.rolling(q_win, min_periods=q_min).sum()  # Replaced 252, 126
    )

    # Build Final Shadow DF
    audit_df = pd.DataFrame(shadow_data, index=ohlcv_to_audit.index)
    print(f"DONE ({time.time()-start_time:.2f}s)")

    # STEP 3: RECONCILIATION REPORT
    print(f"\n{'Metric':<20} | {'Max Delta':<12} | {'Correlation':<12} | {'Status'}")
    print("-" * 85)

    cols_to_check = [
        "Ret_1d",
        "ATR",
        "ATRP",
        "TRP",
        "RSI",
        "Mom_21",
        "Consistency",
        "Beta_63",
        "IR_63",
        "DD_21",
        "RollingStalePct",
        "RollMedDollarVol",
        "RollingSameVolCount",
    ]

    for col in cols_to_check:
        sha_col = f"shadow_{col}"
        if sha_col not in audit_df.columns:
            continue

        eng, sha = features_to_audit[col], audit_df[sha_col]
        # Align and drop NaNs for comparison
        mask = eng.notna() & sha.notna()
        if not mask.any():
            continue

        e_v, s_v = eng[mask], sha[mask]

        delta = (e_v - s_v).abs().max()

        # Suppress the NumPy "Subtract" warning during correlation of constant series
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            # If standard deviation is 0, correlation is undefined; if eng matches Shadow Calculation, we treat as 1.0
            if e_v.std() == 0:
                corr = 1.0 if delta < 1e-6 else 0.0
            else:
                corr = e_v.corr(s_v)

        status = "✅ PASS" if (delta < 1e-6 or corr > 0.99999) else "❌ FAIL"
        print(f"{col:<20} | {delta:>12.4e} | {corr:>12.6f} | {status}")

    vix_z = engine.macro_df["Macro_Vix_Z"].abs().max()
    print(
        f"{'Macro_Vix_Signals':<20} | {'N/A':<12} | {'N/A':<12} | {'✅ LIVE' if vix_z > 0 else '❌ MISSING VIX, VIX3M'}"
    )
    print(f"{'='*95}")


#

In [28]:
# ######################
# # 1. Strip out the NaNs
# # clean_scores = raw_scores.dropna()
# clean_scores = raw_scores
# ######################
audit_feature_engineering_integrity(analyzer1, df_indices=df_indices, mode="system")


🕵️  NUCLEAR FEATURE AUDIT | Mode: SYSTEM | Tickers: 1586
STEP 1: BOUNDARY INTEGRITY   | MultiIndex Isolation Check | ✅ PASS
STEP 2: SHADOW CALCULATIONS  | Re-computing metrics... DONE (109.45s)

Metric               | Max Delta    | Correlation  | Status
-------------------------------------------------------------------------------------
Ret_1d               |   0.0000e+00 |     1.000000 | ✅ PASS
ATR                  |   0.0000e+00 |     1.000000 | ✅ PASS
ATRP                 |   0.0000e+00 |          nan | ✅ PASS
TRP                  |   0.0000e+00 |          nan | ✅ PASS
RSI                  |   0.0000e+00 |     1.000000 | ✅ PASS
Mom_21               |   0.0000e+00 |          nan | ✅ PASS
Consistency          |   0.0000e+00 |     1.000000 | ✅ PASS
Beta_63              |   8.6242e-13 |     1.000000 | ✅ PASS
IR_63                |   0.0000e+00 |     1.000000 | ✅ PASS
DD_21                |   0.0000e+00 |     1.000000 | ✅ PASS
RollingStalePct      |   0.0000e+00 |     1.000000 | ✅ PAS

In [20]:
_tic = "HUBB"
_start_date = "1973-01-08"
_end_date = "1973-02-06"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

               ATR    ATRP  TRP      RSI  Mom_21  Consistency  IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Date                                                                                                                                                 
1973-01-08  0.0980     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                148.0
1973-01-09  0.0910     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                149.0
1973-01-10  0.0845     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                150.0
1973-01-11  0.0931  0.4538  1.0  55.4468     inf          0.0    0.0      1.0 -0.0072     NaN              1.0               0.0                151.0
1973-01-12  0.1011     inf  inf  47.4108 -1.0000          0.0    0.0      1.0 -1.0000    -1.0       

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start="1973-01-08",
    date_end="1973-02-06",
    verbose=False,
)
print(result[_tic])

            Adj Open  Adj High  Adj Low  Adj Close  Volume     ATR    ATRP  TRP      RSI  Mom_21  Consistency  IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Date                                                                                                                                                                                                 
1973-01-08    0.0000    0.0000   0.0000     0.0000       0  0.0980     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                148.0
1973-01-09    0.0000    0.0000   0.0000     0.0000       0  0.0910     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                149.0
1973-01-10    0.0000    0.0000   0.0000     0.0000       0  0.0845     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                150.0
1973-01-11

In [21]:
ohlcv = df_ohlcv.loc[_tic][_start_date:_end_date]
print(ohlcv)

            Adj Open  Adj High  Adj Low  Adj Close  Volume
Date                                                      
1973-01-08    0.0000    0.0000   0.0000     0.0000       0
1973-01-09    0.0000    0.0000   0.0000     0.0000       0
1973-01-10    0.0000    0.0000   0.0000     0.0000       0
1973-01-11    0.2052    0.2052   0.2052     0.2052       0
1973-01-12    0.0000    0.0000   0.0000     0.0000       0
1973-01-15    0.0000    0.0000   0.0000     0.0000       0
1973-01-16    0.2007    0.2007   0.2007     0.2007       0
1973-01-17    0.0000    0.0000   0.0000     0.0000       0
1973-01-18    0.1985    0.1985   0.1985     0.1985       0
1973-01-19    0.0000    0.0000   0.0000     0.0000       0
1973-01-22    0.1962    0.1962   0.1962     0.1962       0
1973-01-23    0.0000    0.0000   0.0000     0.0000       0
1973-01-24    0.0000    0.0000   0.0000     0.0000       0
1973-01-26    0.1798    0.1798   0.1798     0.1798       0
1973-01-29    0.1798    0.1798   0.1798     0.1798      

            Adj Open  Adj High  Adj Low  Adj Close  Volume     ATR    ATRP  TRP      RSI  Mom_21  Consistency  IR_63  Beta_63   DD_21  Ret_1d  RollingStalePct  RollMedDollarVol  RollingSameVolCount
Date                                                                                                                                                                                                 
1973-01-08    0.0000    0.0000   0.0000     0.0000       0  0.0980     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                148.0
1973-01-09    0.0000    0.0000   0.0000     0.0000       0  0.0910     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                149.0
1973-01-10    0.0000    0.0000   0.0000     0.0000       0  0.0845     inf  0.0  47.1247     NaN          0.0    0.0      1.0 -1.0000     NaN              1.0               0.0                150.0
1973-01-11